# 03 Gold: Business Summary Tables
Builds 8 summary tables for the dashboard: by education, age, marital status, product, income, channel, campaign and join year.
- **Output tables:** marketing.default.gold_by_*

In [0]:
from pyspark.sql import functions as F

silver = spark.table("marketing.default.silver_campaign")

# Gold table 1: by education
gold_edu = (silver.groupBy("Education")
    .agg(F.count("*").alias("customers"),
         F.round(F.avg("TotalSpend"), 2).alias("avg_spend"),
         F.round(F.avg("Response") * 100, 2).alias("response_rate_pct")))
gold_edu.write.mode("overwrite").saveAsTable("marketing.default.gold_by_education")

# Gold table 2: by age group
gold_age = (silver.groupBy("AgeGroup")
    .agg(F.count("*").alias("customers"),
         F.round(F.avg("TotalSpend"), 2).alias("avg_spend"),
         F.round(F.avg("Response") * 100, 2).alias("response_rate_pct")))
gold_age.write.mode("overwrite").saveAsTable("marketing.default.gold_by_age")

# Gold table 3: spend by product
products = {"Wines": "MntWines", "Fruits": "MntFruits", "Meat": "MntMeatProducts",
            "Fish": "MntFishProducts", "Sweets": "MntSweetProducts", "Gold": "MntGoldProds"}
rows = [(name, silver.agg(F.sum(col)).first()[0]) for name, col in products.items()]
gold_prod = spark.createDataFrame(rows, ["product", "total_spend"])
gold_prod.write.mode("overwrite").saveAsTable("marketing.default.gold_by_product")

# Gold table 4: by marital status
gold_marital = (silver.groupBy("Marital_Status")
    .agg(F.count("*").alias("customers"),
         F.round(F.avg("TotalSpend"), 2).alias("avg_spend")))
gold_marital.write.mode("overwrite").saveAsTable("marketing.default.gold_by_marital")

# Gold 5: by income group
gold_income = (silver.groupBy("IncomeGroup")
    .agg(F.count("*").alias("customers"),
         F.round(F.avg("TotalSpend"), 2).alias("avg_spend"),
         F.round(F.avg("Response") * 100, 2).alias("response_rate_pct")))
gold_income.write.mode("overwrite").saveAsTable("marketing.default.gold_by_income")

# Gold 6: purchases by channel
channels = {"Web": "NumWebPurchases", "Catalog": "NumCatalogPurchases",
            "Store": "NumStorePurchases", "Deals": "NumDealsPurchases"}
rows = [(n, silver.agg(F.sum(c)).first()[0]) for n, c in channels.items()]
spark.createDataFrame(rows, ["channel", "purchases"]) \
     .write.mode("overwrite").saveAsTable("marketing.default.gold_by_channel")

# Gold 7: acceptance rate per campaign
camps = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5", "Response"]
names = ["Campaign 1", "Campaign 2", "Campaign 3", "Campaign 4", "Campaign 5", "Last Campaign"]
rows = [(n, round(silver.agg(F.avg(c)).first()[0] * 100, 2)) for n, c in zip(names, camps)]
spark.createDataFrame(rows, ["campaign", "acceptance_pct"]) \
     .write.mode("overwrite").saveAsTable("marketing.default.gold_by_campaign")

# Gold 8: customers and spend by join year
gold_year = (silver.groupBy("JoinYear")
    .agg(F.count("*").alias("customers"),
         F.round(F.sum("TotalSpend"), 0).alias("total_spend")))
gold_year.write.mode("overwrite").saveAsTable("marketing.default.gold_by_year")